## SRP358828

**paper:** [PMID: 35500067](https://onlinelibrary.wiley.com/doi/10.1111/1749-4877.12652) - Transcriptome sequencing of black and white hair follicles in the giant panda

**date, curator:** 2026-04-08, Sara Carsanaro

**notes**
* paper says that samples came from 3 separate pandas, seems unlikely they were all 1 year old as stated in SRA especially since they keep calling them adult pandas which most would not consider a 1 year old to be an adult

### annotation summary

In [22]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,hair follicle,UBERON:0002073,hair follicle,perfect match


In [23]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,"1 year, possibly adult, not clear",UBERON:0000066,fully formed stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP358828"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████████| 2/2 [00:01<00:00,  1.08it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX14090267,SRP358828,HiSeq X Ten,SRS11915801,,,,,,hair follicle,"1 year, possibly adult, not clear",,,,,,,9646,,,,,,White hair follicle from gaint panda-T02,SAMN16674361,,,,,,,,white hair follicle,,,08/04/2026,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T02-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR
1,SRX14090266,SRP358828,HiSeq X Ten,SRS11915800,,,,,,hair follicle,"1 year, possibly adult, not clear",,,,,,,9646,,,,,,Black hair follicle from gaint panda -T01,SAMN16674360,,,,,,,,black hair follicle,,,08/04/2026,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T01-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['hair follicle']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0002073'
library.loc[:,'anatName'] = 'hair follicle'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX14090267,SRP358828,HiSeq X Ten,SRS11915801,UBERON:0002073,hair follicle,,,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,,,,,9646,,,,,,White hair follicle from gaint panda-T02,SAMN16674361,,,,,,,,white hair follicle,,,08/04/2026,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T02-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR
1,SRX14090266,SRP358828,HiSeq X Ten,SRS11915800,UBERON:0002073,hair follicle,,,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,,,,,9646,,,,,,Black hair follicle from gaint panda -T01,SAMN16674360,,,,,,,,black hair follicle,,,08/04/2026,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T01-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['1 year, possibly adult, not clear']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000066'
library.loc[:,'stageName'] = 'fully formed stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'other'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX14090267,SRP358828,HiSeq X Ten,SRS11915801,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,,,,9646,,,,,,White hair follicle from gaint panda-T02,SAMN16674361,,,,,,,,white hair follicle,,,08/04/2026,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T02-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR
1,SRX14090266,SRP358828,HiSeq X Ten,SRS11915800,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,,,,9646,,,,,,Black hair follicle from gaint panda -T01,SAMN16674360,,,,,,,,black hair follicle,,,08/04/2026,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T01-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [9]:
library.loc[:,'sex'] = 'NA'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX14090267,SRP358828,HiSeq X Ten,SRS11915801,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,NA,,,9646,,,,,,White hair follicle from gaint panda-T02,SAMN16674361,,,,,,,,white hair follicle,,,08/04/2026,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T02-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR
1,SRX14090266,SRP358828,HiSeq X Ten,SRS11915800,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,NA,,,9646,,,,,,Black hair follicle from gaint panda -T01,SAMN16674360,,,,,,,,black hair follicle,,,08/04/2026,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T01-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR


#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX14090267,SRP358828,HiSeq X Ten,SRS11915801,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,NA,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,White hair follicle from gaint panda-T02,SAMN16674361,,,,,,,,white hair follicle,,,08/04/2026,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T02-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR
1,SRX14090266,SRP358828,HiSeq X Ten,SRS11915800,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,NA,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Black hair follicle from gaint panda -T01,SAMN16674360,,,,,,,,black hair follicle,,,08/04/2026,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T01-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR


#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-08'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX14090267,SRP358828,HiSeq X Ten,SRS11915801,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,NA,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,White hair follicle from gaint panda-T02,SAMN16674361,,,,,,,,white hair follicle,,SAC,2026-04-08,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T02-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR
1,SRX14090266,SRP358828,HiSeq X Ten,SRS11915800,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,NA,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Black hair follicle from gaint panda -T01,SAMN16674360,,,,,,,,black hair follicle,,SAC,2026-04-08,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",,T01-Ailuropodamelanoleura,,,,,,TRANSCRIPTOMIC,RANDOM PCR


#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 35500067, not sure about the 1 year age from SRA, especially since paper says 3 separate adult pandas were used'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX14090267,SRP358828,HiSeq X Ten,SRS11915801,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,NA,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,White hair follicle from gaint panda-T02,SAMN16674361,,,,,,,"PMID: 35500067, not sure about the 1 year age from SRA, especially since paper says 3 separate adult pandas were used",white hair follicle,,SAC,2026-04-08
1,SRX14090266,SRP358828,HiSeq X Ten,SRS11915800,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,NA,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Black hair follicle from gaint panda -T01,SAMN16674360,,,,,,,"PMID: 35500067, not sure about the 1 year age from SRA, especially since paper says 3 separate adult pandas were used",black hair follicle,,SAC,2026-04-08


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP358828,Ailuropoda melanoleuca hair follicle Raw sequence reads,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",SRA,,,,,,,PRJNA804355,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

2

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP358828,Ailuropoda melanoleuca hair follicle Raw sequence reads,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",SRA,total,Bgee 1K,2,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA804355,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '35500067'
experiment.loc[:,'reference_url'] = 'https://onlinelibrary.wiley.com/doi/10.1111/1749-4877.12652'
experiment.loc[:,'DOI'] = '10.1111/1749-4877.12652'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP358828,Ailuropoda melanoleuca hair follicle Raw sequence reads,"The aim of this study was to identify which genes and pathways are involved in determining the differences in panda hair color. We used the RNA-seq technique to identify genes that are differentially expressed in different-colored hair follicles of giant pandas, and we performed functional enrichment analysis of these DEGs. The data set generated in this study can be used as an important resource for genetic research on the fur color of giant pandas and can promote the development of molecular genetics research in giant pandas",SRA,total,Bgee 1K,2,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA804355,35500067,https://onlinelibrary.wiley.com/doi/10.1111/1749-4877.12652,10.1111/1749-4877.12652,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [21]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [24]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [25]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
58950,SRX9681176,SRP297835,Illumina HiSeq 2500,SRS7879028,CL:0000019,sperm,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,mature sperm,sexually mature adult,perfect match,not documented,perfect match,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Giant panda frozen-thawed sperm-rep2,"SAMN17075337,GSM4972354",,,,,,,"PMID: 33969033, this might have issues in the ...",RNA treated by frozen protocol,,SAC,2026-04-08
58951,SRX9681175,SRP297835,Illumina HiSeq 2500,SRS7879027,CL:0000019,sperm,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,mature sperm,sexually mature adult,perfect match,not documented,perfect match,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Giant panda fresh sperm-rep1,"SAMN17075332,GSM4972353",,,,,,,"PMID: 33969033, this might have issues in the ...",,,SAC,2026-04-08
58952,SRX14090267,SRP358828,HiSeq X Ten,SRS11915801,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,NA,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,White hair follicle from gaint panda-T02,SAMN16674361,,,,,,,"PMID: 35500067, not sure about the 1 year age ...",white hair follicle,,SAC,2026-04-08
58953,SRX14090266,SRP358828,HiSeq X Ten,SRS11915800,UBERON:0002073,hair follicle,UBERON:0000066,fully formed stage,,hair follicle,"1 year, possibly adult, not clear",perfect match,not documented,other,NA,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Black hair follicle from gaint panda -T01,SAMN16674360,,,,,,,"PMID: 35500067, not sure about the 1 year age ...",black hair follicle,,SAC,2026-04-08


In [26]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1151,SRP533429,Altered translation elongation contributes to ...,Aging is a major risk factor for neurodegenera...,SRA,partial,Bgee 1K,66,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,GSE277507,PRJNA1162541,40743332,,10.1126/science.adk3079,,
1152,SRP297835,Comparative analysis of piRNA profiles reveals...,"In this study, differentially expressed (DE)pi...",SRA,total,Bgee 1K,4,NEBNext Ultra RNA Library Prep Kit,full_length,GSE163128,PRJNA685032,33969033,https://pmc.ncbi.nlm.nih.gov/articles/PMC8100531/,10.3389/fvets.2021.635013,,this might have issues in the pipeline - polyA...
1153,SRP358828,Ailuropoda melanoleuca hair follicle Raw seque...,The aim of this study was to identify which ge...,SRA,total,Bgee 1K,2,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA804355,35500067,https://onlinelibrary.wiley.com/doi/10.1111/17...,10.1111/1749-4877.12652,,


### add annotations to git

In [27]:
! git pull

Already up to date.


In [28]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [29]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../1_reject_experiment/out/
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [30]:
! git add $git_experiment_path $git_library_path

In [31]:
! git commit -m $commit_message_exp

[develop 1684cce] adding annotated bulk experiment SRP358828
 2 files changed, 3 insertions(+)


In [32]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.16 KiB | 2.16 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   689be65..1684cce  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push